# Glyph-100M: private Colab CUDA workspace
This notebook copies verified archives from Drive to local `/content`, validates base 44k and keeps training disabled until an exact confirmation is entered. It never uses the Drive API.

## A. User configuration

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/Glyph/colab'
RUN_ID = 'glyph100-sft-colab-001'
MANIFEST_FILE = 'manifest.json'
SHA256SUMS_FILE = 'SHA256SUMS'
CODE_ARCHIVE = 'glyph-code-YYYYMMDDTHHMMSSZ.tar.zst'
TOKENIZER_ARCHIVE = 'glyph-tokenizer-YYYYMMDDTHHMMSSZ.tar.zst'
BASE_CHECKPOINT_ARCHIVE = 'glyph-base44k-YYYYMMDDTHHMMSSZ.tar.zst'
DATASET_ARCHIVE = 'glyph-sft-data-YYYYMMDDTHHMMSSZ.tar.zst'
REPORTS_ARCHIVE = ''

RUN_PREFLIGHT = True
RUN_LABEL_AUDIT = True
RUN_INFERENCE_SMOKE = True
RUN_COMPATIBILITY_SMOKE = False
RUN_TRAINING = False
RUN_EVAL = False
EXPORT_RESULTS = False

REUSE_LOCAL_FILES = False
ALLOW_PYTORCH_REINSTALL = False
PYTORCH_INSTALL_COMMAND = []  # Fill only after documenting a real incompatibility.
TRAINING_CONFIRMATION = ''
EXPORT_CONFIRMATION = ''
RESUME_CHECKPOINT = ''

LOCAL_CODE_ROOT = '/content/glyph'
LOCAL_INPUT_ROOT = '/content/glyph-input'
LOCAL_OUTPUT_ROOT = '/content/glyph-output'
BASE_CHECKPOINT = f'{LOCAL_CODE_ROOT}/checkpoints/glyph-100m-base44k.pt'
TOKENIZER = f'{LOCAL_CODE_ROOT}/data/processed/tokenizer.model'
DATASET = f'{LOCAL_CODE_ROOT}/data/sft/glyph100_sft_smoke_v0_3.jsonl'
TRAIN_JSONL = f'{LOCAL_CODE_ROOT}/data/sft/processed/glyph100_sft_smoke_v0_3_train.jsonl'
VAL_JSONL = f'{LOCAL_CODE_ROOT}/data/sft/processed/glyph100_sft_smoke_v0_3_val.jsonl'
RUN_ROOT = f'{LOCAL_OUTPUT_ROOT}/{RUN_ID}'
PREFLIGHT_ROOT = f'{LOCAL_OUTPUT_ROOT}/{RUN_ID}-preflight'

MAX_STEPS = 400
BATCH_SIZE = 1
GRAD_ACCUM = 4
LEARNING_RATE = 1e-5
PRECISION = 'fp32'

## B. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## C. Runtime information

In [ ]:
import json, os, platform, shutil, subprocess, sys
import torch

runtime = {
    'python': sys.version,
    'pytorch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'cuda': torch.version.cuda,
    'cudnn': torch.backends.cudnn.version() if torch.cuda.is_available() else None,
    'platform': platform.platform(),
    'content_disk': shutil.disk_usage('/content')._asdict(),
}
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    free_vram, total_vram = torch.cuda.mem_get_info(0)
    runtime['gpu'] = {
        'name': props.name,
        'compute_capability': f'{props.major}.{props.minor}',
        'free_vram': free_vram,
        'total_vram': total_vram,
        'bf16_supported': torch.cuda.is_bf16_supported(),
    }
try:
    import psutil
    runtime['ram'] = psutil.virtual_memory()._asdict()
except ImportError:
    runtime['ram'] = 'psutil will be installed later'
print(json.dumps(runtime, indent=2, default=str))
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before continuing.')

## D. Copy, SHA256-check and safely extract bundles locally

In [ ]:
import hashlib, json, os, shutil, subprocess, sys, tarfile
from pathlib import Path, PurePosixPath

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'zstandard>=0.22,<1'])
import zstandard as zstd

drive_input = Path(DRIVE_ROOT) / 'input'
local_input = Path(LOCAL_INPUT_ROOT)
local_code = Path(LOCAL_CODE_ROOT)
local_input.mkdir(parents=True, exist_ok=True)
if local_code.exists() and any(local_code.iterdir()) and not REUSE_LOCAL_FILES:
    raise FileExistsError(f'{local_code} is not empty. Set REUSE_LOCAL_FILES only after inspection.')
local_code.mkdir(parents=True, exist_ok=True)

def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(4 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

manifest_source = drive_input / MANIFEST_FILE
sums_source = drive_input / SHA256SUMS_FILE
if not manifest_source.is_file():
    raise FileNotFoundError(manifest_source)
if not sums_source.is_file():
    raise FileNotFoundError(sums_source)
manifest_local = local_input / MANIFEST_FILE
sums_local = local_input / SHA256SUMS_FILE
shutil.copy2(manifest_source, manifest_local)
shutil.copy2(sums_source, sums_local)
outer_sums = {}
for line in sums_local.read_text().splitlines():
    if line.strip():
        digest, name = line.split(None, 1)
        outer_sums[name.strip()] = digest
if outer_sums.get(MANIFEST_FILE) != sha256(manifest_local):
    raise ValueError('manifest.json does not match SHA256SUMS')
manifest = json.loads(manifest_local.read_text())
by_name = {item['path']: item for item in manifest['archives']}
requested = [CODE_ARCHIVE, TOKENIZER_ARCHIVE, BASE_CHECKPOINT_ARCHIVE, DATASET_ARCHIVE]
if REPORTS_ARCHIVE:
    requested.append(REPORTS_ARCHIVE)

def safe_extract_tar_zst(archive_path, destination):
    root = destination.resolve()
    with open(archive_path, 'rb') as source:
        with zstd.ZstdDecompressor().stream_reader(source) as reader:
            with tarfile.open(fileobj=reader, mode='r|') as archive:
                for member in archive:
                    pure = PurePosixPath(member.name)
                    if pure.is_absolute() or '..' in pure.parts or member.issym() or member.islnk() or member.isdev():
                        raise ValueError(f'Unsafe archive member: {member.name}')
                    target = (destination / pure.as_posix()).resolve()
                    if root != target and root not in target.parents:
                        raise ValueError(f'Archive escape: {member.name}')
                    if member.isdir():
                        target.mkdir(parents=True, exist_ok=True)
                    elif member.isfile():
                        target.parent.mkdir(parents=True, exist_ok=True)
                        extracted = archive.extractfile(member)
                        if extracted is None:
                            raise IOError(member.name)
                        with open(target, 'wb') as output:
                            shutil.copyfileobj(extracted, output, length=4 * 1024 * 1024)
                    else:
                        raise ValueError(f'Unsupported archive member: {member.name}')

for name in requested:
    if name not in by_name:
        raise KeyError(f'{name} is not listed in manifest.json')
    source = drive_input / name
    if not source.is_file():
        raise FileNotFoundError(source)
    destination = local_input / name
    if destination.exists() and not REUSE_LOCAL_FILES:
        raise FileExistsError(destination)
    if not destination.exists():
        shutil.copy2(source, destination)
    actual = sha256(destination)
    expected = by_name[name]['sha256']
    if actual != expected:
        raise ValueError(f'SHA256 mismatch for {name}: {actual} != {expected}')
    if outer_sums.get(name) != actual:
        raise ValueError(f'{name} does not match SHA256SUMS')
    safe_extract_tar_zst(destination, local_code)
    print('verified and extracted', name)

print('Local Glyph runtime:', local_code)

## E. Install minimal dependencies without replacing CUDA PyTorch

In [ ]:
import subprocess, sys, torch
if not torch.cuda.is_available():
    if not ALLOW_PYTORCH_REINSTALL:
        raise RuntimeError('Current PyTorch cannot see CUDA. No automatic reinstall is allowed.')
    if not PYTORCH_INSTALL_COMMAND:
        raise RuntimeError('Document the incompatibility and provide PYTORCH_INSTALL_COMMAND explicitly.')
    print('Explicit PyTorch replacement requested because CUDA is unavailable:', PYTORCH_INSTALL_COMMAND)
    subprocess.check_call(PYTORCH_INSTALL_COMMAND)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{LOCAL_CODE_ROOT}/requirements-colab.txt'])
print('PyTorch retained:', torch.__version__, 'CUDA:', torch.version.cuda)

## F/G. Preflight, label audit and inference smoke (no training)

In [ ]:
import subprocess, sys
if RUN_PREFLIGHT or RUN_LABEL_AUDIT or RUN_INFERENCE_SMOKE:
    cmd = [
        sys.executable, f'{LOCAL_CODE_ROOT}/scripts/colab_preflight.py',
        '--run-id', RUN_ID,
        '--checkpoint', BASE_CHECKPOINT,
        '--tokenizer', TOKENIZER,
        '--dataset', DATASET,
        '--reports-dir', f'{PREFLIGHT_ROOT}/reports',
        '--expected-step', '44000',
        '--expected-variant', 'glyph-100m',
        '--expected-context', '512',
        '--expected-dataset', 'glyph100_v2_3_1',
        '--precision', PRECISION,
    ]
    if not RUN_LABEL_AUDIT:
        cmd.append('--skip-label-audit')
    if not RUN_INFERENCE_SMOKE:
        cmd.append('--skip-inference')
    print(' '.join(cmd))
    subprocess.check_call(cmd)
else:
    print('Preflight disabled by user configuration.')

## H. Optional 20-step CUDA SFT compatibility smoke

In [ ]:
import subprocess, sys
if RUN_COMPATIBILITY_SMOKE:
    if TRAINING_CONFIRMATION != 'RUN_COMPATIBILITY_SMOKE':
        raise RuntimeError('Set TRAINING_CONFIRMATION exactly to RUN_COMPATIBILITY_SMOKE.')
    compat_id = f'{RUN_ID}-compat'
    cmd = [
        sys.executable, f'{LOCAL_CODE_ROOT}/scripts/colab_run_sft.py',
        '--run-id', compat_id, '--base-checkpoint', BASE_CHECKPOINT,
        '--train-jsonl', TRAIN_JSONL, '--val-jsonl', VAL_JSONL, '--tokenizer', TOKENIZER,
        '--output-dir', f'{LOCAL_OUTPUT_ROOT}/{compat_id}', '--max-steps', '20',
        '--batch-size', '1', '--gradient-accumulation-steps', '4',
        '--learning-rate', '1e-5', '--min-lr', '1e-6', '--weight-decay', '0',
        '--grad-clip', '0.5', '--eval-interval', '10', '--checkpoint-interval', '10',
        '--log-interval', '1', '--precision', 'fp32',
        '--dataset-name', 'glyph100_sft_colab_compatibility_smoke', '--confirm-training',
    ]
    print('Starting explicitly confirmed compatibility smoke only.')
    subprocess.check_call(cmd)
else:
    print('Compatibility smoke is disabled.')

## I. Manually approved SFT run or resume

In [ ]:
import subprocess, sys
if RUN_TRAINING:
    if TRAINING_CONFIRMATION != 'START_GLYPH_SFT':
        raise RuntimeError('Set TRAINING_CONFIRMATION exactly to START_GLYPH_SFT.')
    cmd = [
        sys.executable, f'{LOCAL_CODE_ROOT}/scripts/colab_run_sft.py',
        '--run-id', RUN_ID, '--base-checkpoint', BASE_CHECKPOINT,
        '--train-jsonl', TRAIN_JSONL, '--val-jsonl', VAL_JSONL, '--tokenizer', TOKENIZER,
        '--output-dir', RUN_ROOT, '--max-steps', str(MAX_STEPS),
        '--batch-size', str(BATCH_SIZE), '--gradient-accumulation-steps', str(GRAD_ACCUM),
        '--learning-rate', str(LEARNING_RATE), '--min-lr', '1e-6', '--weight-decay', '0',
        '--grad-clip', '0.5', '--eval-interval', '50', '--checkpoint-interval', '100',
        '--log-interval', '5', '--precision', PRECISION,
        '--dataset-name', 'glyph100_sft_colab', '--confirm-training',
    ]
    if RESUME_CHECKPOINT:
        cmd.extend(['--resume-checkpoint', RESUME_CHECKPOINT])
    print('Starting explicitly confirmed run:', ' '.join(cmd))
    subprocess.check_call(cmd)
else:
    print('RUN_TRAINING=False; no optimizer or backward pass was started.')

## Optional post-run CUDA reload/eval smoke

In [ ]:
import subprocess, sys, torch
from pathlib import Path
if RUN_EVAL:
    candidate = Path(RUN_ROOT) / 'checkpoints' / 'best.pt'
    if not candidate.is_file():
        candidate = Path(RUN_ROOT) / 'checkpoints' / 'latest.pt'
    if not candidate.is_file():
        raise FileNotFoundError('No best/latest checkpoint to evaluate.')
    state = torch.load(candidate, map_location='cpu', weights_only=False)
    step = int(state.get('current_step', state.get('step', 0)))
    cmd = [sys.executable, f'{LOCAL_CODE_ROOT}/scripts/colab_preflight.py',
           '--run-id', f'{RUN_ID}-postrun', '--checkpoint', str(candidate),
           '--tokenizer', TOKENIZER, '--dataset', DATASET,
           '--reports-dir', f'{RUN_ROOT}/reports', '--expected-step', str(step),
           '--expected-variant', 'glyph-100m', '--expected-context', '512',
           '--expected-dataset', '', '--precision', PRECISION]
    subprocess.check_call(cmd)
else:
    print('Post-run eval is disabled.')

## J. Validate, archive locally, then optionally copy result to Drive

In [ ]:
import subprocess, sys
if EXPORT_RESULTS:
    if EXPORT_CONFIRMATION != 'COPY_VALIDATED_RESULT':
        raise RuntimeError('Set EXPORT_CONFIRMATION exactly to COPY_VALIDATED_RESULT.')
    cmd = [sys.executable, f'{LOCAL_CODE_ROOT}/scripts/colab_export_results.py',
           '--run-dir', RUN_ROOT, '--run-id', RUN_ID, '--tokenizer', TOKENIZER,
           '--output-dir', f'{LOCAL_OUTPUT_ROOT}/exports',
           '--copy-to', f'{DRIVE_ROOT}/output',
           '--confirm-copy', 'COPY_VALIDATED_RESULT']
    subprocess.check_call(cmd)
else:
    print('EXPORT_RESULTS=False; nothing was copied to Drive.')